In [1]:
# 🎯 Objectif du projet
#
# Ce projet a pour objectif d’automatiser le téléchargement, le traitement et le stockage
# des données publiques du service Yellow Taxi de New York (fichiers .parquet).
# L’idée est de construire une pipeline locale permettant de :
#
# 📥 Télécharger automatiquement les fichiers Parquet correspondant à plusieurs périodes (mois/années).
#
# 🧹 Charger et filtrer uniquement les colonnes pertinentes (distance, montant, nombre de passagers, etc.).
#
# 🗃️ Stocker les données dans une base de données SQLite structurée, pour une exploitation ultérieure
# (analyse statistique, modélisation, data visualisation).
#
# 🧩 Garantir l’intégrité des données en évitant le rechargement des fichiers déjà insérés
# grâce à une colonne de suivi (source_file).


In [2]:
import os
import requests
import sqlite3
import pandas as pd 
from pathlib import Path


La phase Extract

In [3]:
# Definir le dossier de téléchargent en local

DATA_DIR = Path("yellow_taxi_data")

# Au cas où le dossier n'existe pas je vais le créer

DATA_DIR.mkdir(exist_ok=True)

# Liste des fichiers parquet à télécharger

years = [2023, 2024]
months = range(1, 3) # Janvier et fevrier

for year in years:
    for month in months:
        file_name = f"yellow_tripdata_{year}-{month:02d}.parquet"
        file_path = DATA_DIR/file_name
        
        if file_path.exists():
            print(f"{file_name} déja téléchargé.")
            continue
        
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file_name}"
        response = requests.get(url)
        
        if response.status_code == 200:
            with open(file_path, "wb") as f:
                f.write(response.content)
            print(f"✅ Téléchargé : {file_name}")
        else:
            print(f"❌ Echec de téléchargement : {file_name}")
            
        

✅ Téléchargé : yellow_tripdata_2023-01.parquet
✅ Téléchargé : yellow_tripdata_2023-02.parquet
✅ Téléchargé : yellow_tripdata_2024-01.parquet
✅ Téléchargé : yellow_tripdata_2024-02.parquet


La phase LOAD

L'objectif est que tous les fichiers doivent résidés dans une base de données SQLite

In [4]:
# Chemin vers le fichier parquet
file_path = Path("yellow_taxi_data/yellow_tripdata_2024-01.parquet")

# Lire le fichier parquet avec pandas
df = pd.read_parquet(file_path)

# Vérifier le début du DataFrame
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [5]:
# Afficher la structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [6]:

import sqlite3

# Dossier où stocker la base
DB_DIR = Path("yellow_taxi_data")  
DB_DIR.mkdir(exist_ok=True)

DB_PATH = DB_DIR / "trips.db"  # chemin complet vers le fichier SQLite

# Connexion à SQLite
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("✅ Base SQLite prête :", DB_PATH.resolve())

# Liste des colonnes à récupérer depuis les fichiers Parquet
expected_columns = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
    "trip_distance", "RatecodeID", "store_and_fwd_flag", "PULocationID",
    "DOLocationID", "payment_type", "fare_amount", "extra", "mta_tax",
    "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
    "congestion_surcharge", "Airport_fee"
]

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()

    # Supprimer l'ancienne table si elle existe
    cursor.execute("DROP TABLE IF EXISTS trips")

    # Recréer proprement la table avec en ajoutant la colonne source_file
    cursor.execute("""
    CREATE TABLE trips (
        VendorID INTEGER,
        tpep_pickup_datetime TEXT,
        tpep_dropoff_datetime TEXT,
        passenger_count REAL,
        trip_distance REAL,
        RatecodeID REAL,
        store_and_fwd_flag TEXT,
        PULocationID INTEGER,
        DOLocationID INTEGER,
        payment_type INTEGER,
        fare_amount REAL,
        extra REAL,
        mta_tax REAL,
        tip_amount REAL,
        tolls_amount REAL,
        improvement_surcharge REAL,
        total_amount REAL,
        congestion_surcharge REAL,
        Airport_fee REAL,
        source_file TEXT
    )
    """)
    conn.commit()
    print(f"✅ Table 'trips' créée avec succès dans {DB_PATH.resolve()}")

    # Charger la liste des fichiers déjà importés (vide pour la 1ʳᵉ exécution)
    existing_files = set()

    # Charger les fichiers parquet dans SQLite
    for file_path in DATA_DIR.glob("*.parquet"):
        if file_path.name in existing_files:
            print(f"Fichier déjà chargé, passage : {file_path.name}")
            continue # On passe au fichier suivant
    
        df = pd.read_parquet(file_path, engine="pyarrow")
    
        # Sélectionner des colonnes disponibles dans le fichier
        available_columns = list(set(df.columns) & set(expected_columns)) # Intersection
    
        df = df[available_columns] # On sélectionne uniquement les colonnes existantes
    
        # Convertir des types pour correspondre à SQLite parceque SQLite ne gére de maniére native les formats datetime
        if "tpep_pickup_datetime" in df.columns:
            df["tpep_pickup_datetime"] = df["tpep_pickup_datetime"].astype(str)
        
        if "tpep_dropoff_datetime" in df.columns:
            df["tpep_dropoff_datetime"] = df["tpep_dropoff_datetime"].astype(str)
    
        if "store_and_fwd_flag" in  df.columns:
            df["store_and_fwd_flag"] = df["store_and_fwd_flag"].astype(str)
        
        # Ajout le nom du fichier source
        df["source_file"] = file_path.name
    
        # Insertion dans la base trips SQLite
        df.to_sql("trips", conn, if_exists="append", index=False)
        print(f"Données inserées depuis : {file_path.name}")
    
    # Verification des données
    print(pd.read_sql("SELECT COUNT(*) AS total_lignes FROM trips", conn))
    
# Fermeture de la connexion
conn.close()

✅ Base SQLite prête : C:\Users\savan\OneDrive\Bureau\Data\.venv\yellow_taxi_data\trips.db
✅ Table 'trips' créée avec succès dans C:\Users\savan\OneDrive\Bureau\Data\.venv\yellow_taxi_data\trips.db
Données inserées depuis : yellow_tripdata_2023-01.parquet
Données inserées depuis : yellow_tripdata_2023-02.parquet
Données inserées depuis : yellow_tripdata_2024-01.parquet
Données inserées depuis : yellow_tripdata_2024-02.parquet
   total_lignes
0      11952871


In [7]:
# Verifier que le chargementest est bien fait, sans doublon

# Le dossier contenant les fichiers parquet
DATA_DIR = Path("yellow_taxi_data")
DB_PATH = DATA_DIR / "trips.db"

# Connexion à la base SQLite
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Comptage du nombre total de lignes dans les fichiers parquets
total_rows_parquet = sum(pd.read_parquet(file, engine="pyarrow").shape[0] for file in DATA_DIR.glob("*.parquet"))

# Comptage du nombre de lignes dans la table SQLite
total_rows_SQLite = pd.read_sql("SELECT COUNT(*) AS total FROM trips", conn)["total"][0]

# Fermeture de la connexion
conn.close()

# Affichage des résultats
print(f"total des lignes dans les fichiers PARQUET : {total_rows_parquet}")
print(f"Total de lignes dans la table trips (SQLite) : {total_rows_SQLite}")

# Verification
if total_rows_parquet == total_rows_SQLite:
    print("✅ Les nombres de lignes correspondent")
else:
    print("⚠️ Il y'a une difference entre les fichiers parquet et la base SQLite")

total des lignes dans les fichiers PARQUET : 11952871
Total de lignes dans la table trips (SQLite) : 11952871
✅ Les nombres de lignes correspondent
